# Data Cleaning and Preparation

## Objective

Create an analysis-ready version of the salary dataset while preserving the original raw data.

## Decisions from Data Understanding

- Exact matching rows will be retained because the dataset has no unique respondent identifier.
- Statistical salary outliers will be retained because high salaries may be valid.
- `salary_in_usd` will be used for salary comparisons.
- Original coded columns will be preserved, and readable label columns will be added.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
project_root = Path.cwd().parent

raw_file_path = (
    project_root / "data" / "raw" / "salaries.csv"
)

processed_file_path = (
    project_root / "data" / "processed" / "salaries_clean.csv"
)

raw_file_path, processed_file_path

(WindowsPath('c:/Users/aydna/global-ai-ml-salary-analysis/data/raw/salaries.csv'),
 WindowsPath('c:/Users/aydna/global-ai-ml-salary-analysis/data/processed/salaries_clean.csv'))

In [3]:
df_raw = pd.read_csv(raw_file_path)

df_clean = df_raw.copy()

df_raw.shape, df_clean.shape

((151445, 11), (151445, 11))

## Text Quality Checks

Check text columns for unnecessary leading or trailing spaces before applying any modifications.

In [4]:
text_columns = (
    df_clean
    .select_dtypes(include="object")
    .columns
    .tolist()
)

text_columns

['experience_level',
 'employment_type',
 'job_title',
 'salary_currency',
 'employee_residence',
 'company_location',
 'company_size']

In [5]:
whitespace_counts = {}

for column in text_columns:
    cleaned_text = df_clean[column].str.strip()

    rows_with_spaces = (
        df_clean[column] != cleaned_text
    ).sum()

    whitespace_counts[column] = rows_with_spaces

pd.Series(
    whitespace_counts, 
    name = "rows_whith-extra_spaces"
)

experience_level      0
employment_type       0
job_title             0
salary_currency       0
employee_residence    0
company_location      0
company_size          0
Name: rows_whith-extra_spaces, dtype: int64

### Whitespace Check Result

No leading or trailing spaces were detected in the text columns. Therefore, no whitespace cleaning was required.

In [6]:
experience_map = {
    "EN": "Entry-level",
    "MI": "Mid-level",
    "SE": "Senior-level",
    "EX": "Executive-level"
}

employment_map = {
    "FT": "Full-time",
    "PT": "Part-time",
    "CT": "Contract",
    "FL": "Freelance"
}

company_size_map = {
    "S": "Small",
    "M": "Medium",
    "L": "Large"
}

remote_map = {
    0: "On-site",
    50: "Hybrid",
    100: "Remote"
}

In [20]:
df_clean["experience_level_label"] = (
    df_clean["experience_level"].map(experience_map)
)

df_clean["employment_type_label"] = (
    df_clean["employment_type"].map(employment_map)
)

df_clean["company_size_label"] = (
    df_clean["company_size"].map(company_size_map)
)

df_clean["remote_type"] = (
    df_clean["remote_ratio"].map(remote_map)
)    

In [21]:
df_clean[[

    "experience_level",
        "experience_level_label",
        "employment_type",
        "employment_type_label",
        "company_size",
        "company_size_label",
        "remote_ratio",
        "remote_type"
]].head(10)

,experience_level,experience_level_label,employment_type,employment_type_label,company_size,company_size_label,remote_ratio,remote_type
0,EX,Executive-level,FT,Full-time,M,Medium,0,On-site
1,EX,Executive-level,FT,Full-time,M,Medium,0,On-site
2,SE,Senior-level,FT,Full-time,M,Medium,0,On-site
3,SE,Senior-level,FT,Full-time,M,Medium,0,On-site
4,MI,Mid-level,FT,Full-time,M,Medium,100,Remote
5,MI,Mid-level,FT,Full-time,M,Medium,100,Remote
6,SE,Senior-level,FT,Full-time,M,Medium,100,Remote
7,SE,Senior-level,FT,Full-time,M,Medium,100,Remote
8,SE,Senior-level,FT,Full-time,M,Medium,100,Remote
9,SE,Senior-level,FT,Full-time,M,Medium,100,Remote


In [22]:
label_columns = [

    "experience_level_label",
    "employment_type_label",
    "company_size_label",
    "remote_type"
]

df_clean[label_columns].isna().sum()

experience_level_label    0
employment_type_label     0
company_size_label        0
remote_type               0
dtype: int64

### Categorical Labeling Result

Readable label columns were successfully created for experience level, employment type, company size, and remote-work type.

No unmapped categorical codes were detected. The original coded columns were retained for traceability.

In [23]:
column_order = [
     "work_year",
    "experience_level",
    "experience_level_label",
    "employment_type",
    "employment_type_label",
    "job_title",
    "salary",
    "salary_currency",
    "salary_in_usd",
    "employee_residence",
    "remote_ratio",
    "remote_type",
    "company_location",
    "company_size",
    "company_size_label"
]

df_clean = df_clean[column_order]

df_clean.head()

,work_year,experience_level,experience_level_label,employment_type,employment_type_label,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,remote_type,company_location,company_size,company_size_label
0,2025,EX,Executive-level,FT,Full-time,Head of Data,348516,USD,348516,US,0,On-site,US,M,Medium
1,2025,EX,Executive-level,FT,Full-time,Head of Data,232344,USD,232344,US,0,On-site,US,M,Medium
2,2025,SE,Senior-level,FT,Full-time,Data Scientist,145400,USD,145400,US,0,On-site,US,M,Medium
3,2025,SE,Senior-level,FT,Full-time,Data Scientist,81600,USD,81600,US,0,On-site,US,M,Medium
4,2025,MI,Mid-level,FT,Full-time,Engineer,160000,USD,160000,US,100,Remote,US,M,Medium


In [24]:
final_check = {

    "raw_rows": len(df_raw),
    "clean_rows": len(df_clean),
    "raw_columns": df_raw.shape[1],
    "clean_columns": df_clean.shape[1],
    "missing_values": df_clean.isna().sum().sum(),
    "duplicate_rows_after_first": df_clean.duplicated().sum()

}

pd.Series(final_check)

raw_rows                      151445
clean_rows                    151445
raw_columns                       11
clean_columns                     15
missing_values                     0
duplicate_rows_after_first     79532
dtype: int64

In [25]:
df_clean.to_csv(
    processed_file_path,
    index=False
)

In [26]:
df_saved = pd.read_csv(processed_file_path)

df_saved.shape

(151445, 15)

## Cleaning Summary

- The raw dataset was preserved without modification.
- No rows were removed.
- No missing values or unnecessary whitespace were detected.
- Exact matching rows were retained due to the absence of a unique respondent identifier.
- Statistical salary outliers were retained because they may represent valid high-paying roles.
- Four readable categorical label columns were added.
- The cleaned dataset contains 151,445 rows and 15 columns.
- The final dataset was saved to `data/processed/salaries_clean.csv`.